<a href="https://colab.research.google.com/github/galihtw04/data-science-2026/blob/master/Pertemuan12_Muhammad_Galih_Abdurrahman_240401010278.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 12

**Nama:** Muhammad Galih Abdurrahman  
**NIM:** 240401010278  
**Kelas:** IF403

---

## Tujuan Pembelajaran

Pada pertemuan ini, kita akan belajar tentang:
1. **Market Basket Analysis**: Cara menemukan pola pembelian produk yang sering dibeli bersamaan
2. **Association Rules**: Membuat aturan asosiasi untuk merekomendasikan produk
3. **Content-Based Filtering**: Sistem rekomendasi berdasarkan kesamaan karakteristik produk
4. **Perbandingan Pendekatan**: Memahami perbedaan antara dua metode rekomendasi

---

## Pengenalan Market Basket Analysis

**Market Basket Analysis** adalah teknik untuk menemukan produk-produk yang sering dibeli bersamaan oleh pelanggan. Misalnya:
- Orang yang beli **Roti** biasanya juga beli **Selai**
- Orang yang beli **Kopi** biasanya juga beli **Gula**

Teknik ini sangat berguna untuk:
- **Penempatan produk** di toko (taruh produk terkait berdekatan)
- **Rekomendasi produk** di e-commerce
- **Promosi bundling** (jual paket produk yang sering dibeli bersamaan)

### Istilah Penting:

1. **Support**: Seberapa sering produk muncul dalam transaksi
   - Contoh: Jika Roti muncul di 16 dari 50 transaksi, maka support = 16/50 = 0.32 atau 32%

2. **Confidence**: Seberapa sering aturan "Jika beli A, maka beli B" benar
   - Contoh: Dari 16 orang yang beli Roti, 11 orang juga beli Selai, maka confidence = 11/16 = 0.69 atau 69%

3. **Lift**: Seberapa kuat hubungan antara dua produk dibanding jika acak
   - Lift > 1: Produk saling terkait (beli A meningkatkan kemungkinan beli B)
   - Lift = 1: Tidak ada hubungan khusus
   - Lift < 1: Produk saling menghindari (beli A menurunkan kemungkinan beli B)

### Algoritma Apriori

**Apriori** adalah algoritma untuk menemukan frequent itemset (kelompok produk yang sering muncul bersamaan). Cara kerjanya:
1. Hitung support dari setiap produk
2. Buang produk yang support-nya terlalu kecil (jarang dibeli)
3. Gabungkan produk-produk yang tersisa untuk membentuk pasangan, triplet, dst
4. Hitung support dari setiap kombinasi
5. Buang kombinasi yang support-nya kecil

---

## Sistem Rekomendasi (Recommendation System)

**Sistem Rekomendasi** adalah sistem yang memberikan saran produk kepada pengguna. Ada beberapa pendekatan:

### 1. Content-Based Filtering

Merekomendasikan produk berdasarkan **kesamaan karakteristik** produk:
- Jika kamu suka Roti, maka kami rekomendasikan Sereal dan Selai (karena sama-sama kategori Bakery)
- Jika kamu suka Film Action, kami rekomendasikan Film Action lainnya

**Keuntungan:**
- Tidak butuh data pengguna lain
- Bisa menjelaskan kenapa produk direkomendasikan ("karena sama kategorinya")

**Kekurangan:**
- Rekomendasi terbatas pada produk serupa (kurang eksplorasi)
- Butuh informasi detail tentang produk

### 2. Collaborative Filtering (Association Rules)

Merekomendasikan produk berdasarkan **pola pembelian pengguna lain**:
- "Orang yang beli Roti biasanya juga beli Selai, jadi kami rekomendasikan Selai untuk kamu"

**Keuntungan:**
- Bisa menemukan hubungan yang tidak terduga (misalnya orang yang beli popok bayi ternyata sering beli bir)
- Tidak perlu tahu detail produk

**Kekurangan:**
- Butuh banyak data transaksi
- Tidak bisa merekomendasikan produk baru yang belum pernah dibeli

### 3. Hybrid System

Menggabungkan kedua pendekatan untuk mendapatkan hasil terbaik.

---

## Hands-On: Market Basket Analysis

Mari kita praktikkan dengan data transaksi minimarket!

### Task 1: Generate & Eksplorasi Dataset Transaksi

Kita akan membuat data transaksi simulasi dengan 10 produk yang berbeda.

In [18]:
# Generate & Eksplorasi Dataset Transaksi
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

produk = [
    'Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
    'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'
]

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []

for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
        list(np.random.choice(produk, n_item, replace=False))
    )

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


### Task 2: One-Hot Encoding Transaksi

Sebelum menggunakan algoritma Apriori, kita perlu mengubah data transaksi menjadi format **one-hot encoding**:
- `True` berarti produk ada dalam transaksi
- `False` berarti produk tidak ada dalam transaksi

Ini seperti mengubah data dari:
- Transaksi 1: ["Roti", "Selai", "Kopi"]

Menjadi:
- Transaksi 1: Roti=True, Selai=True, Kopi=True, Susu=False, Telur=False, ...

Format ini memudahkan komputer untuk menghitung support dan menemukan pola.

In [19]:
# One-Hot Encoding Transaksi
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


### Task 3: Cari Frequent Itemset dengan Apriori

Sekarang kita akan mencari **frequent itemset** menggunakan algoritma Apriori.

**Min_support** adalah threshold minimum untuk support:
- Jika min_support = 0.1, artinya produk harus muncul minimal di 10% transaksi (5 dari 50 transaksi)
- Jika min_support terlalu kecil (misal 0.05), kita dapat terlalu banyak itemset termasuk yang tidak penting
- Jika min_support terlalu besar (misal 0.5), kita hanya dapat produk yang sangat populer saja

Kita akan coba beberapa nilai min_support untuk menemukan nilai yang pas.

In [20]:
# Cari Frequent Itemset dengan Apriori
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
   freq = apriori(df, min_support=ms, use_colnames=True)
   print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


### Task 4: Bentuk & Saring Aturan Asosiasi

Dari frequent itemset, kita akan membentuk **aturan asosiasi** dalam format:
- **"Jika beli A, maka beli B"**

Aturan ini memiliki metrik:
- **Confidence**: Seberapa sering aturan benar (minimum 0.5 atau 50%)
- **Lift**: Seberapa kuat hubungan antara A dan B (harus > 1 untuk menunjukkan ada korelasi positif)

Kita akan mengurutkan aturan berdasarkan **Lift tertinggi** untuk menemukan aturan terkuat.

In [21]:
# Bentuk & Saring Aturan Asosiasi
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
'support', 'confidence', 'lift']].head(10))
# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

         antecedents consequents  support  confidence      lift
9        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
13  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
8      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
15   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


**Interpretasi Aturan Asosiasi:**

Dari hasil di atas, aturan dengan **Lift tertinggi** adalah:
1. **"Jika beli Keju dan Teh → maka beli Telur"** (Lift = 2.38, Confidence = 86%)
   - Artinya: Orang yang beli Keju dan Teh, 2.38 kali lebih mungkin beli Telur dibanding orang random
   - Ini aturan paling kuat!

2. **"Jika beli Mentega dan Selai → maka beli Kopi"** (Lift = 1.95, Confidence = 63%)
   - Masuk akal: Orang yang beli bahan sarapan (Mentega, Selai) biasanya beli minuman (Kopi)

3. **"Jika beli Roti dan Gula → maka beli Selai"** (Lift = 1.92, Confidence = 100%)
   - Sangat masuk akal: Roti, Gula, dan Selai adalah bahan sarapan yang sering dibeli bersamaan

4. **"Jika beli Roti → maka beli Selai"** (Lift = 1.32, Confidence = 69%)
   - Ini pola klasik yang kita suntikkan ke data! Roti dan Selai memang sering dibeli bersamaan

**Insight Bisnis:**
- Taruh Roti dan Selai berdekatan di toko
- Buat promosi bundling "Paket Sarapan: Roti + Selai + Kopi"
- Di e-commerce, rekomendasikan Selai ketika customer beli Roti

### Task 5: Rekomender Sederhana dengan Content-Based Filtering

Sekarang kita akan membuat sistem rekomendasi menggunakan pendekatan **Content-Based Filtering**.

Ide dasar:
- Setiap produk memiliki karakteristik (misal kategori: Bakery, Dairy, Minuman, Bumbu)
- Kita menghitung **kesamaan** antar produk berdasarkan kategori menggunakan **Cosine Similarity**
- Jika kamu suka produk A, kami rekomendasikan produk yang kategorinya mirip dengan A

**Cosine Similarity** mengukur seberapa mirip dua produk:
- Nilai 1.0 = sangat mirip (kategori sama persis)
- Nilai 0.0 = tidak mirip sama sekali (kategori berbeda total)

In [22]:
# Rekomender Sederhana dengan Content-Based Filtering
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
'produk': produk,
'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
   idx = katalog.index[katalog['produk'] == nama_produk][0]
   skor = list(enumerate(sim_matrix[idx]))
   skor = sorted(skor, key=lambda x: x[1], reverse=True)
   skor = [s for s in skor if s[0] != idx][:top_n]
   return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


### Task 6: Bandingkan Kedua Pendekatan

Mari kita bandingkan rekomendasi dari **Association Rules** vs **Content-Based Filtering** untuk produk "Roti".

In [23]:
# Bandingkan Kedua Pendekatan
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


**Perbandingan Hasil:**

**Association Rules merekomendasikan:**
- Selai (Lift = 1.92 dan 1.32)

**Content-Based Filtering merekomendasikan:**
- Selai, Sereal, Susu (karena kategorinya mirip dengan Roti)

**Kesimpulan:**
- Kedua metode sama-sama merekomendasikan **Selai** → ini bagus karena konsisten!
- Association Rules hanya fokus pada pola pembelian nyata ("orang yang beli Roti biasanya beli Selai")
- Content-Based memberikan rekomendasi lebih banyak berdasarkan kesamaan kategori (Bakery dan Dairy)

**Kapan menggunakan masing-masing?**

1. **Gunakan Association Rules** jika:
   - Punya banyak data transaksi
   - Ingin menemukan pola pembelian yang tidak terduga
   - Fokus pada "apa yang sering dibeli bersamaan"

2. **Gunakan Content-Based Filtering** jika:
   - Data transaksi masih sedikit
   - Punya informasi detail tentang produk (kategori, merek, harga, dll)
   - Ingin rekomendasikan produk baru yang belum pernah dibeli

3. **Gunakan Hybrid System** (gabungan keduanya) jika:
   - Ingin mendapatkan hasil terbaik dari kedua metode
   - Contoh: Rekomendasikan Selai (dari Association Rules) DAN Sereal (dari Content-Based) untuk customer yang beli Roti

---

## Kesimpulan

Pada pertemuan ini, saya telah belajar tentang **Market Basket Analysis** dan **Sistem Rekomendasi** untuk menemukan pola pembelian dan merekomendasikan produk.

### Poin-Poin Penting:

1. **Market Basket Analysis menggunakan algoritma Apriori** untuk menemukan produk yang sering dibeli bersamaan. Pada **Task 1**, saya membuat 50 transaksi dengan 10 produk berbeda (Roti, Selai, Susu, dll).

2. **One-Hot Encoding** (Task 2) mengubah data transaksi menjadi format True/False agar mudah diproses komputer.

3. **Frequent Itemset** (Task 3) ditemukan dengan min_support = 0.1, menghasilkan 44 itemset. Produk paling populer adalah Selai (52%), Teh (46%), dan Mentega (42%).

4. **Association Rules** (Task 4) menghasilkan aturan seperti "Jika beli Keju dan Teh → beli Telur" dengan Lift = 2.38 (aturan terkuat), dan "Jika beli Roti → beli Selai" dengan Lift = 1.32.

5. **Content-Based Filtering** (Task 5) merekomendasikan produk berdasarkan kesamaan kategori menggunakan Cosine Similarity. Untuk Roti (Bakery), sistem merekomendasikan Selai, Sereal, dan Susu.

6. **Perbandingan** (Task 6) menunjukkan bahwa Association Rules dan Content-Based sama-sama merekomendasikan Selai untuk Roti, tapi Content-Based memberikan opsi lebih banyak.

### Aplikasi dalam Bisnis:

- **Retail**: Taruh produk yang sering dibeli bersamaan berdekatan di toko
- **E-commerce**: "Customers who bought this also bought..." menggunakan Association Rules
- **Marketing**: Buat promosi bundling berdasarkan frequent itemset
- **Inventory**: Stok produk yang sering dibeli bersamaan agar tidak kehabisan

Teknik ini sangat berguna untuk meningkatkan penjualan dan memberikan pengalaman belanja yang lebih baik kepada pelanggan!